# Danuj's Python + OOP Notebook (Code Collection)
**Contents**
1. List, Tuple, Dict, Set, Frozenset
2. Mutable vs Immutable + Traps
3. Encapsulation and Abstraction
4. Why `__init__`
5. Class Variables and `@classmethod`
6. `@staticmethod` + Full Program + Overriding
7. Extras: dunder methods, inheritance preview, Two Sum, practice solutions

> Run all: **Kernel -> Restart & Run All**. Java code is shown as markdown (Java cannot run in a Python kernel).

# 1. Collections
## 1.1 List (ordered, mutable)

In [ ]:
nums = [10, 20, 30, 40]
mixed = [1, "Danuj", 3.14, True]

print(nums[0])       # 10
print(nums[-1])      # 40 (negative index = from end)
print(nums[1:3])     # [20, 30]  slice [start:stop:step], stop exclusive
print(nums[::-1])    # [40, 30, 20, 10] reverse (new list)

nums[0] = 99
nums.append(50)      # O(1)
nums.insert(1, 15)   # O(n) shifting
nums.remove(30)      # O(n)
last = nums.pop()    # O(1)
print(nums, "popped:", last)

### List comprehension
`[expression for item in iterable if condition]`

In [ ]:
squares = [x * x for x in range(1, 6)]
evens = [x for x in range(10) if x % 2 == 0]
print(squares)   # [1, 4, 9, 16, 25]
print(evens)     # [0, 2, 4, 6, 8]

## 1.2 Tuple (ordered, immutable)

In [ ]:
point = (10, 20)
single = (5,)        # comma required
not_tuple = (5)      # just an int
print(type(single), type(not_tuple))

try:
    point[0] = 99
except TypeError as e:
    print("TypeError:", e)

x, y = point                 # unpacking
a, b = 1, 2
a, b = b, a                  # swap
print(x, y, a, b)

visited = {}
visited[(2, 3)] = True       # tuple as dict key
try:
    visited[[2, 3]] = True
except TypeError as e:
    print("TypeError:", e)

## 1.3 Dict (hash map)

In [ ]:
student = {"name": "Danuj", "age": 21, "skills": ["Java", "Python"]}

print(student["name"])
print(student.get("city"))          # None
print(student.get("city", "NA"))    # NA

student["city"] = "Himatnagar"
del student["age"]

for key, value in student.items():
    print(key, "->", value)

try:
    student["xyz"]
except KeyError as e:
    print("KeyError:", e)

### Frequency counting: manual, Counter, defaultdict

In [1]:
from collections import Counter, defaultdict

arr = [1, 2, 2, 3, 3, 3]

freq = {}
for n in arr:
    freq[n] = freq.get(n, 0) + 1
print(freq)                         # {1: 1, 2: 2, 3: 3}

print(Counter(arr))
print(Counter(arr).most_common(1))  # [(3, 3)]

groups = defaultdict(list)          # anagram grouping
for word in ["eat", "tea", "tan", "ate", "nat"]:
    groups["".join(sorted(word))].append(word)
print(dict(groups))

{1: 1, 2: 2, 3: 3}
Counter({3: 3, 2: 2, 1: 1})
[(3, 3)]
{'aet': ['eat', 'tea', 'ate'], 'ant': ['tan', 'nat']}


## 1.4 Set and Frozenset

In [ ]:
nums = {1, 2, 3, 3, 2}
print(nums)                         # {1, 2, 3}

empty_set = set()
empty_dict = {}
print(type(empty_set), type(empty_dict))

unique = set([1, 2, 2, 3, 3, 3])
print(unique)

s = {10, 20, 30}
s.add(40)
s.add(10)
s.remove(20)
s.discard(99)                       # safe
print(30 in s, len(s), s)

try:
    s.remove(12345)
except KeyError as e:
    print("remove() KeyError:", e)

In [ ]:
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}
print("union       ", a | b)
print("intersection", a & b)
print("difference  ", a - b)
print("symmetric   ", a ^ b)
print("subset      ", {1, 2} <= a)
print("disjoint    ", a.isdisjoint({7, 8}))

print({x * x for x in range(-3, 4)})   # set comprehension

fs = frozenset([1, 2, 3])
d = {fs: "group1"}
print(d)
try:
    fs.add(4)
except AttributeError as e:
    print("AttributeError:", e)

### DSA: Contains Duplicate (LeetCode 217) + intersection

In [ ]:
def contains_duplicate(nums):
    seen = set()
    for n in nums:
        if n in seen:
            return True
        seen.add(n)
    return False

print(contains_duplicate([1, 2, 3, 1]))   # True
print(contains_duplicate([1, 2, 3]))      # False
print(set([1, 2, 2, 3]) & set([2, 2, 3, 4]))   # {2, 3}

### Java equivalents
```java
Map<Integer, Integer> freq = new HashMap<>();
for (int n : arr) freq.put(n, freq.getOrDefault(n, 0) + 1);

public boolean containsDuplicate(int[] nums) {
    Set<Integer> seen = new HashSet<>();
    for (int n : nums) if (!seen.add(n)) return true;
    return false;
}
```

# 2. Mutable vs Immutable: operations and traps

In [ ]:
l = [1, 2, 3]
l.append(4); l.insert(0, 0); l.remove(2); l.pop(); l[0] = 99
l.sort(); l.reverse()
print("list ->", l)

t = (1, 2, 3)
t2 = t + (4,)            # new tuple
print("tuple ->", t, t2)

s = {1, 2, 3}
s.add(4); s.discard(10); s |= {7, 8}
print("set ->", s)

d = {"a": 1}
d["b"] = 2; d["a"] = 100; d.pop("b"); d.setdefault("c", 0); d.update({"x": 5})
print("dict ->", d)

### Trap 1: mutable object inside tuple

In [ ]:
t = (1, [2, 3])
t[1].append(4)
print(t)      # (1, [2, 3, 4])

### Trap 2: aliasing

In [ ]:
a = [1, 2, 3]
b = a
b.append(4)
print(a)                 # [1, 2, 3, 4]
c = a.copy()
print(a is b, a is c)    # True False

### Trap 3: shallow vs deep copy

In [ ]:
import copy
x = [[1, 2], [3, 4]]
y = x.copy()
z = copy.deepcopy(x)
x[0].append(99)
print("shallow:", y)
print("deep   :", z)

### Trap 4: mutable default argument

In [ ]:
def add_bad(x, lst=[]):
    lst.append(x)
    return lst

print(add_bad(1), add_bad(2))     # [1, 2] [1, 2]  same list!

def add_good(x, lst=None):
    if lst is None:
        lst = []
    lst.append(x)
    return lst

print(add_good(1), add_good(2))   # [1] [2]

### Trap 5: 2D list bug

In [ ]:
grid_bad = [[0] * 3] * 3
grid_bad[0][0] = 1
print(grid_bad)      # all rows changed

grid_good = [[0] * 3 for _ in range(3)]
grid_good[0][0] = 1
print(grid_good)

### `is` vs `==`, id()

In [ ]:
p = [1, 2]; q = [1, 2]
print(p == q, p is q, id(p) == id(q))

# 3. Encapsulation and Abstraction (Python)
## 3.1 Encapsulation

In [ ]:
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self.__balance = balance        # name mangling

    @property
    def balance(self):
        return self.__balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        self.__balance += amount

    def withdraw(self, amount):
        if amount > self.__balance:
            raise ValueError("Insufficient balance")
        self.__balance -= amount

acc = BankAccount("Danuj", 1000)
acc.deposit(500)
print(acc.balance)                       # 1500
try:
    print(acc.__balance)
except AttributeError as e:
    print("AttributeError:", e)
print(acc._BankAccount__balance)         # still reachable (mangled)
try:
    acc.deposit(-5)
except ValueError as e:
    print("ValueError:", e)

### Property with setter validation

In [ ]:
class Student:
    def __init__(self, marks):
        self.marks = marks                # goes through setter

    @property
    def marks(self):
        return self._marks

    @marks.setter
    def marks(self, value):
        if not 0 <= value <= 100:
            raise ValueError("marks must be 0-100")
        self._marks = value

s = Student(80)
s.marks = 95
print(s.marks)
try:
    s.marks = 150
except ValueError as e:
    print("ValueError:", e)

## 3.2 Abstraction (abc)

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

class Circle(Shape):
    def __init__(self, r):
        self.r = r
    def area(self):
        return 3.14159 * self.r ** 2

class Rectangle(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h
    def area(self):
        return self.w * self.h

try:
    Shape()
except TypeError as e:
    print("TypeError:", e)

for sh in [Circle(5), Rectangle(4, 6)]:      # polymorphism
    print(type(sh).__name__, sh.area())

### Practice: abstract Vehicle

In [ ]:
class Vehicle(ABC):
    @abstractmethod
    def start(self):
        pass

class Car(Vehicle):
    def start(self): print("Car starts with key")

class Bike(Vehicle):
    def start(self): print("Bike starts with kick")

for v in (Car(), Bike()):
    v.start()

### Java versions
```java
public class BankAccount {
    private String owner;
    private double balance;
    public BankAccount(String owner, double balance) { this.owner = owner; this.balance = balance; }
    public double getBalance() { return balance; }
    public void deposit(double amount) {
        if (amount <= 0) throw new IllegalArgumentException("Amount must be positive");
        balance += amount;
    }
    public void withdraw(double amount) {
        if (amount > balance) throw new IllegalStateException("Insufficient balance");
        balance -= amount;
    }
}

abstract class Shape {
    String color;
    Shape(String color) { this.color = color; }
    abstract double area();
    void showColor() { System.out.println("Color: " + color); }
}
class Circle extends Shape {
    double radius;
    Circle(String color, double radius) { super(color); this.radius = radius; }
    @Override double area() { return Math.PI * radius * radius; }
}

interface Payment { void pay(double amount); }
class UpiPayment implements Payment { public void pay(double a) { System.out.println("UPI " + a); } }
```

# 4. Why `__init__`?

In [ ]:
class Circle:
    def __init__(self, r):
        self.r = r
    def area(self):
        return 3.14159 * self.r ** 2
    def perimeter(self):
        return 2 * 3.14159 * self.r

c = Circle(5)
print(c.area(), c.perimeter())

c1, c2 = Circle(5), Circle(10)
print(c1.area(), c2.area())

### Without `__init__` -> error

In [ ]:
class CircleNoInit:
    def area(self):
        return 3.14159 * self.r ** 2

try:
    CircleNoInit().area()
except AttributeError as e:
    print("AttributeError:", e)

### Practice: Rectangle with validation

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        if width < 0 or height < 0:
            raise ValueError("width/height cannot be negative")
        self.width = width
        self.height = height
    def area(self): return self.width * self.height
    def perimeter(self): return 2 * (self.width + self.height)

r = Rectangle(4, 6)
print(r.area(), r.perimeter())
try:
    Rectangle(-1, 5)
except ValueError as e:
    print("ValueError:", e)

```java
class Circle {
    double r;
    Circle(double r) { this.r = r; }
    double area() { return Math.PI * r * r; }
}
Circle c = new Circle(5);
```

# 5. Class variables and `@classmethod`

In [ ]:
class Student:
    class_year = 2024                 # class variable (shared)
    def __init__(self, name, age):
        self.name = name              # instance variables
        self.age = age

student1 = Student("Spongebob", 30)
student2 = Student("Patrick", 35)
print(student2.name)
print(student1.age)
print(student1.class_year)

In [ ]:
class Student:
    class_year = 2024
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def get_year(cls):
        return cls.class_year

    @classmethod
    def set_year(cls, new_year):
        cls.class_year = new_year

    @classmethod
    def from_string(cls, data):
        name, age = data.split("-")
        return cls(name, int(age))

s1 = Student("Spongebob", 30)
s2 = Student("Patrick", 35)
print(Student.get_year())
Student.set_year(2025)
print(s1.class_year, s2.class_year)

p = Student.from_string("Patrick-35")
print(p.name, p.age)

### Shadowing trap

In [ ]:
s1.class_year = 2030
print(s1.class_year, s2.class_year, Student.class_year)   # 2030 2025 2025
print(s1.__dict__)    # class_year now lives in the instance

### Practice: counting objects (and the `self.count += 1` bug)

In [ ]:
class Counter1:
    count = 0
    def __init__(self):
        Counter1.count += 1
    @classmethod
    def get_count(cls):
        return cls.count

[Counter1() for _ in range(3)]
print("correct:", Counter1.get_count())     # 3

class Counter2:
    count = 0
    def __init__(self):
        self.count += 1          # BUG: creates instance variable each time

[Counter2() for _ in range(3)]
print("buggy  :", Counter2.count)           # 0

```java
class Student {
    static int classYear = 2024;
    String name; int age;
    Student(String name, int age) { this.name = name; this.age = age; }
    static void setYear(int y) { classYear = y; }
}
```

# 6. `@staticmethod`, full program, overriding

In [ ]:
class MathHelper:
    @staticmethod
    def add(a, b): return a + b
    @staticmethod
    def is_even(n): return n % 2 == 0

print(MathHelper.add(3, 4), MathHelper.is_even(10))

### Validator (real-life static methods)

In [ ]:
class Validator:
    @staticmethod
    def is_valid_email(email):
        return "@" in email and "." in email
    @staticmethod
    def is_strong_password(pwd):
        return len(pwd) >= 8 and any(c.isdigit() for c in pwd)

print(Validator.is_valid_email("danuj@gmail.com"))   # True
print(Validator.is_strong_password("abc123"))        # False

### One complete program: instance + classmethod + staticmethod

In [ ]:
class Student:
    school_name = "Sunrise School"
    total_students = 0

    def __init__(self, name, marks):
        self.name = name
        self.marks = marks
        Student.total_students += 1

    def show(self):
        print(f"{self.name} | Marks: {self.marks} | School: {Student.school_name}")

    @classmethod
    def get_total(cls): return cls.total_students

    @classmethod
    def change_school(cls, new_name): cls.school_name = new_name

    @classmethod
    def from_string(cls, data):
        name, marks = data.split("-")
        return cls(name, int(marks))

    @staticmethod
    def is_pass(marks): return marks >= 33

    @staticmethod
    def grade(marks):
        if marks >= 90: return "A+"
        elif marks >= 75: return "A"
        elif marks >= 33: return "B"
        return "Fail"

s1 = Student("Danuj", 92)
s2 = Student("Rahul", 28)
s3 = Student.from_string("Amit-80")
s1.show(); s2.show(); s3.show()
print("Total:", Student.get_total())
Student.change_school("Galaxy School")
s1.show()
print(Student.is_pass(s2.marks))
print(Student.grade(s1.marks))

### Practice: Temperature (static methods)

In [ ]:
class Temperature:
    @staticmethod
    def celsius_to_fahrenheit(c): return c * 9 / 5 + 32
    @staticmethod
    def fahrenheit_to_celsius(f): return (f - 32) * 5 / 9

print(Temperature.celsius_to_fahrenheit(100))   # 212.0
print(Temperature.fahrenheit_to_celsius(32))    # 0.0

### Overriding static methods

In [ ]:
class Student:
    @staticmethod
    def is_pass(marks): return marks >= 33

class TopperStudent(Student):
    @staticmethod
    def is_pass(marks): return marks >= 75

print(Student.is_pass(50), TopperStudent.is_pass(50))   # True False

#### The catch: hardcoded parent call, and two fixes

In [ ]:
class Student:
    @staticmethod
    def is_pass(marks): return marks >= 33

    def result_bad(self, marks):
        return Student.is_pass(marks)        # hardcoded parent

    def result_good(self, marks):
        return self.is_pass(marks)           # Fix 1: self

    @classmethod
    def check(cls, marks):
        return cls.is_pass(marks)            # Fix 2: cls

class TopperStudent(Student):
    @staticmethod
    def is_pass(marks): return marks >= 75

t = TopperStudent()
print("bad  :", t.result_bad(50))             # True (surprise)
print("self :", t.result_good(50))            # False
print("cls  :", TopperStudent.check(50))      # False
print("MRO  :", [c.__name__ for c in TopperStudent.__mro__])

### Practice: Shape.describe override + classmethod info

In [ ]:
class Shape:
    @staticmethod
    def describe(): return "I am a shape"
    @classmethod
    def info(cls): return cls.describe()

class Circle(Shape):
    @staticmethod
    def describe(): return "I am a circle"

print(Shape.info())     # I am a shape
print(Circle.info())    # I am a circle

### Java: method hiding
```java
class Student { static boolean isPass(int m) { return m >= 33; } }
class Topper extends Student { static boolean isPass(int m) { return m >= 75; } }
Student s = new Topper();
System.out.println(s.isPass(50));   // true -> reference type decides (hiding, not overriding)
```

# 7. Extras
## 7.1 Dunder methods

In [ ]:
class Point:
    def __init__(self, x, y): self.x, self.y = x, y
    def __str__(self):  return f"({self.x}, {self.y})"
    def __repr__(self): return f"Point({self.x}, {self.y})"
    def __eq__(self, o): return self.x == o.x and self.y == o.y
    def __hash__(self): return hash((self.x, self.y))
    def __add__(self, o): return Point(self.x + o.x, self.y + o.y)
    def __len__(self): return 2

p = Point(1, 2) + Point(3, 4)
print(p, repr(p), len(p))
print(len({Point(1, 2), Point(1, 2)}))    # 1

## 7.2 Inheritance + polymorphism preview

In [ ]:
class Animal:
    def speak(self): return "..."
class Dog(Animal):
    def speak(self): return "Woof"
class Cat(Animal):
    def speak(self): return "Meow"

for a in [Dog(), Cat(), Animal()]:
    print(type(a).__name__, "->", a.speak())

## 7.3 Two Sum: brute force vs hash map

In [ ]:
def two_sum_brute(nums, target):          # O(n^2)
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]

def two_sum(nums, target):                # O(n)
    seen = {}
    for i, n in enumerate(nums):
        need = target - n
        if need in seen:
            return [seen[need], i]
        seen[n] = i

print(two_sum_brute([2, 7, 11, 15], 9))
print(two_sum([2, 7, 11, 15], 9))
print(two_sum([3, 2, 4], 6))

```java
public int[] twoSum(int[] nums, int target) {
    Map<Integer, Integer> seen = new HashMap<>();
    for (int i = 0; i < nums.length; i++) {
        int need = target - nums[i];
        if (seen.containsKey(need)) return new int[]{seen.get(need), i};
        seen.put(nums[i], i);
    }
    return new int[]{};
}
```

## 7.4 Practice: Longest Consecutive Sequence (LeetCode 128) with a set

In [ ]:
def longest_consecutive(nums):
    s = set(nums)
    best = 0
    for n in s:
        if n - 1 not in s:          # n is the start of a sequence
            length = 1
            while n + length in s:
                length += 1
            best = max(best, length)
    return best

print(longest_consecutive([100, 4, 200, 1, 3, 2]))   # 4

## 7.5 Time complexity demo: list vs set `in`

In [ ]:
import time
N = 20000
lst = list(range(N)); st = set(lst)
targets = list(range(N - 2000, N))

t0 = time.perf_counter(); [x in lst for x in targets]; t1 = time.perf_counter()
[x in st for x in targets]; t2 = time.perf_counter()
print(f"list: {t1 - t0:.4f}s | set: {t2 - t1:.6f}s")

**Done!** Next: Inheritance, Polymorphism, then DSA problems in Java.